# Tidal corroboration of the 2025 missing-flood events

**Read this first — what this notebook is for.**

Earlier work found a *data gap*: dozens of flood "bumps" are visible on the floodnet.nyc
depth graphs during 2025 but are **missing from the public flood-events feed**
(`aq7i-eu5q`). You logged those missing events by hand in
`FloodNet_Data_Manual_QC.csv`. A separate notebook (`Weather_Data_Scrape.ipynb`) already
checked whether each missing event had a **rainfall** driver nearby.

This notebook answers the *complementary* question: **was the flood tidal — or driven by
storm surge — rather than rain?** It is the tidal analog of the rain check.

---

**Vocabulary (important — this trips people up):**
- A **sensor** is a physical FloodNet device at a street corner. There are **11** unique
  sensors in the QC log.
- An **event** is one *(date, sensor)* pair — "sensor X showed a missing flood on day Y."
  There are **226** events across those 11 sensors (227 hand-logged rows minus 1 dropped —
  see below). Most numbers below count *events*, not sensors. (So when you see "164", that
  is 164 *events*, not 164 sensors.)
- One QC row (`SI - Grimsby St/Mapleton Ave`, 2025-12-19) is **dropped entirely**: the
  automated rain-driver check (built in `Weather_Data_Scrape.ipynb`) found it's already
  sitting in the public feed on that exact date — a hand-log slip, not a genuinely missing
  event — so it doesn't belong in either the rain or no-rain bucket.

---

**Why a tide check cannot be built the same way as the rain check.**
Rain is *episodic* — most days have none — so "did it rain that day?" is a discriminating
question. But a high tide happens **twice every single day**, so "was there a high tide?"
is *always yes* and tells you nothing. A tidal signal therefore has to be measured by
**magnitude**, not presence. Two magnitude questions:

1. **Astronomical tide** — did the flood fall on a day whose high tide was *unusually high*
   (a spring / "king" tide)? We answer this with a **percentile**: where does that day's
   highest high water rank among all days of the year at that station?
2. **Surge** — was the water pushed *above* the astronomical prediction by weather (wind /
   low pressure)? We answer this with **surge = observed − astronomical** at the high.
   A no-rain coastal flood driven by surge is a real, distinct category.

**Granularity — why this is a *day-level* test.** The QC log records the *date* of each
missing event, not a clock time. Tides cycle ~twice a day, so without a timestamp we
cannot measure *lag to the high-tide hour* (the "phase" test in `tidal_analysis.ipynb`).
So this notebook tests tide **magnitude at day resolution**, and each event is bucketed as
**tidal**, **surge**, **compound**, or **neither**. See the final "What this shows" cell
for what that can and cannot conclude.

## 0. Data inputs

Everything below is built from files that already exist in the repo — this notebook
fetches only NOAA observed water levels new (cached, 2025-only).

| file | what it is | produced by |
|---|---|---|
| `FloodNet_Data_Manual_QC.csv` | your hand log of 2025 flood bumps seen on the graph but missing from the public feed — one row per date, sensors comma-separated | manual review |
| `qc_rain_driver_check.csv` | automated per-event rain-driver check (rain at the nearest station on the flood day or the day before) + feed-verification status | `Weather_Data_Scrape.ipynb` |
| `final_deployed_sensors.geojson` | the 415 deployed sensors with lat/lon | `FloodNet_Sensor_Locations.ipynb` |
| `tidal_analysis/tidal_unified.geojson` | high/low tides for all 6 stations, 2025–2026 (NOAA predicted + USGS observed with surge); **trimmed to 2025 in memory here** — the shared file itself is untouched since `tidal_analysis.ipynb` wants the full 2025–2026 range | `tidal_data_format_conversion.ipynb` |
| `noaa_observed_water_levels.csv` | **fetched here** — 6-min NOAA observed levels, 2025 only, cached (~9 MB) | this notebook, cell 3 |

**Outputs it writes:** `noaa_observed_water_levels.csv` (cache) and
`tidal_corroboration_events.csv` (the full per-event classification, for editing).

In [1]:
# =====================================================================================
# SETUP + LOAD THE MANUAL QC LOG
# =====================================================================================
import json
from pathlib import Path
from math import radians, sin, cos, asin, sqrt

import numpy as np
import pandas as pd
import requests

# --- File paths. This notebook lives in data_gap_exploration/; REPO is the repo root. ---
BASE        = Path.cwd()                                   # .../data_gap_exploration
REPO        = BASE.parent                                  # repo root
TIDAL       = REPO / "tidal_analysis" / "tidal_unified.geojson"   # H/L tides, all 6 stations
QC          = BASE / "FloodNet_Data_Manual_QC.csv"         # your hand-logged missing events
RAIN_CHECK  = BASE / "qc_rain_driver_check.csv"             # automated rain-driver check (Weather_Data_Scrape.ipynb)
SENSORS_GEO = REPO / "final_deployed_sensors.geojson"      # sensor lat/lon
NOAA_CACHE  = BASE / "noaa_observed_water_levels.csv"      # cached NOAA observed fetch (~14 MB)

# Re-fetch NOAA observed water levels from the API even if the cache already exists.
# Leave False for normal runs; the cache makes re-runs instant.
FORCE_REFRESH = False

# --- Classification thresholds. These are DELIBERATE CHOICES, exposed here so you can ---
# --- tune them. The enrichment test later is threshold-free and is the more robust read. --
ASTRO_PCTILE_THRESHOLD = 0.80   # day's highest high water in the top 20% of the year => "king/spring tide"
SURGE_FT_THRESHOLD     = 1.0    # >= 1 ft of positive surge at the high => "surge"

# --- Load the QC log and EXPLODE it to one row per (date, sensor) ---------------------
# The raw CSV has one row per date, with a comma-separated "Sensors" cell listing every
# sensor that showed a missing flood that day. .str.split(",") + .explode() turns each of
# those into its own row, so downstream everything is per-(date, sensor) = per "event".
qc = pd.read_csv(QC)
ev = qc.assign(sensor_name=qc["Sensors"].str.split(",")).explode("sensor_name")
ev["sensor_name"]  = ev["sensor_name"].str.strip()                     # tidy whitespace
ev["date"]         = pd.to_datetime(ev["Date"], format="%m/%d/%Y").dt.date

# --- Attach the AUTOMATED rain-driver check (built in Weather_Data_Scrape.ipynb) ------
# rain_flag: True if there was rain at the sensor's nearest weather station on the event
# day or the day before. This replaces the earlier version of this notebook, which used
# the manual QC "Weather" column -- a much cruder hand tag that disagreed with this one.
# The check also flags QC rows that are actually already IN the public feed (a hand-log
# slip, not a real missing event) -- those are DROPPED here rather than counted as rain,
# since they aren't genuinely-missing events to tidally classify at all.
rain_check = pd.read_csv(RAIN_CHECK, parse_dates=["date"])
rain_check["date"] = rain_check["date"].dt.date
ev = ev.merge(rain_check[["date", "sensor_name", "status"]], on=["date", "sensor_name"], how="left")

qc_slip_mask = ev["status"].str.startswith("in feed", na=False)
if qc_slip_mask.any():
    print(f"dropping {qc_slip_mask.sum()} QC row(s) already present in the public feed "
          f"(hand-log slip, not a genuinely missing event):")
    print(ev.loc[qc_slip_mask, ["date", "sensor_name", "status"]].to_string(index=False))
ev = ev[~qc_slip_mask].copy()

ev["rain_flag"] = ev["status"].eq("genuinely missing")

ev = ev[["date", "sensor_name", "rain_flag"]].reset_index(drop=True)

# Print the shape in plain language so it is unambiguous what each count means.
print(f"{len(ev)} EVENTS (date-sensor pairs) across {ev.sensor_name.nunique()} unique SENSORS")
print(f"  of those {len(ev)} event-rows: {ev.rain_flag.sum()} rain-driven (automated check), "
      f"{(~ev.rain_flag).sum()} not")

dropping 1 QC row(s) already present in the public feed (hand-log slip, not a genuinely missing event):
      date                   sensor_name            status
2025-12-19 SI - Grimsby St/ Mapleton Ave in feed (on time)
226 EVENTS (date-sensor pairs) across 11 unique SENSORS
  of those 226 event-rows: 62 rain-driven (automated check), 164 not


## 1. Assign each sensor its nearest tidal station

Each flood must be compared to the tides *near it*. There are 6 tidal stations in the
dataset (2 NOAA harmonic-prediction gauges + 4 USGS observed gauges). For each of the 11
sensors we pick the **closest** station by straight-line (haversine) distance — the same
nearest-neighbour approach `tidal_analysis.ipynb` uses. Station coordinates come from the
tide file; sensor coordinates from the deployed-sensors GeoJSON.

In [2]:
# =====================================================================================
# NEAREST TIDAL STATION PER SENSOR
# =====================================================================================
tu_json = json.load(open(TIDAL))
feats   = tu_json["features"]          # list of GeoJSON features, one per tide reading

# Collapse the tide file to ONE point per station (id -> name, lat, lon).
# Every feature repeats its station's coords, so we just keep the first of each id.
_st = {}
for f in feats:
    p = f["properties"]
    _st[str(p["station_id"])] = (p["station_name"],
                                 f["geometry"]["coordinates"][1],   # GeoJSON is [lon, lat]
                                 f["geometry"]["coordinates"][0])
stations = pd.DataFrame([(k, *v) for k, v in _st.items()],
                        columns=["station_id", "station_name", "lat", "lon"])

# Sensor coordinates, keyed by sensor_name, from the deployed-sensors GeoJSON.
sd = json.load(open(SENSORS_GEO))
scoord = {f["properties"].get("sensor_name"):
          (f["geometry"]["coordinates"][1], f["geometry"]["coordinates"][0])
          for f in sd["features"] if f["geometry"]}

def haversine_km(la1, lo1, la2, lo2):
    """Great-circle distance in km between two lat/lon points."""
    dlat, dlon = radians(la2 - la1), radians(lo2 - lo1)
    a = sin(dlat / 2) ** 2 + cos(radians(la1)) * cos(radians(la2)) * sin(dlon / 2) ** 2
    return 2 * 6371 * asin(sqrt(a))     # 6371 km = Earth radius

# For each sensor, compute distance to all 6 stations and keep the nearest.
rows = []
for s in ev.sensor_name.unique():
    la, lo = scoord[s]
    d = stations.apply(lambda r: haversine_km(la, lo, r.lat, r.lon), axis=1)
    n = stations.loc[d.idxmin()]
    rows.append((s, n.station_id, n.station_name, round(d.min(), 2)))
smap = pd.DataFrame(rows, columns=["sensor_name", "station_id", "station_name", "dist_km"])

# Attach the chosen station to every event.
ev = ev.merge(smap, on="sensor_name")

# 7 of 11 sensors sit on USGS stations (surge available directly); 4 on NOAA (surge fetched below).
smap.sort_values("station_name")

,sensor_name,station_id,station_name,dist_km
0,SI - McLaughlin St/Agnes Pl,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,7.39
1,SI - Grimsby St/ Mapleton Ave,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5.05
4,SI - Baden Pl/ Mapleton Ave,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5.07
6,SI - Hylan Blvd/ Jefferson Blvd,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5.39
10,SI - Bedford Ave/Kiswick St,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5.00
2,Q - Beach Channel Dr/Beach 48th St,01311850,JAMAICA BAY AT INWOOD NY,3.08
3,Q - Beach 49th St/Rockaway Beach Blvd,01311850,JAMAICA BAY AT INWOOD NY,3.27
5,BX - Watson Ave/Close Ave,8516945,Kings Point,10.03
7,BX - Tier St/William Ave,8516945,Kings Point,4.68
8,SI - Snug Harbor Rd / Kissel Ave,8518750,The Battery,9.99


### Why the station split matters

The 6 stations come in two flavors, and which one a sensor is nearest to decides how much
work its surge takes:

- **4 USGS gauges** (Great Kills, Jamaica Bay, Rockaway Inlet, East Rockaway) record *observed*
  water level, and `tidal_unified.geojson` already carries their surge (`surge_ft`). **7 of the
  11 sensors** are nearest one of these — surge is free for them.
- **2 NOAA gauges** (The Battery, Kings Point) provide *predictions* only in our file. **4
  sensors** (2 Bronx + Minthorne + Snug Harbor) are nearest these, so cell 3 fetches NOAA
  observed water levels to compute their surge.

The end result is one uniform high-tide table where every high — NOAA or USGS — has both an
astronomical height and a surge, defined the same way.

## 2. Build the high-tide table: astronomical height + surge at every high

We only care about **high** tides (`tidal_phase == "H"`). For each high we need two numbers:
its **astronomical** height (the pure tide, surge removed) and the **surge** at that moment.
The two station types store this differently:

- **USGS** (Great Kills, Jamaica Bay, Rockaway Inlet, East Rockaway): `water_level_ft` is the
  *observed* level and `surge_ft` is already computed (= observed − UTide-reconstructed
  astronomical). So **astronomical = `water_level_ft − surge_ft`**, and surge is taken as-is.
- **NOAA** (The Battery, Kings Point): `water_level_ft` *is* the astronomical prediction and
  `surge_ft` is 0 (no observed data in this file). We fill in NOAA surge in the next cell.

Both sources are already in the same vertical datum (NAVD88 feet), so heights are comparable.

In [3]:
# =====================================================================================
# HIGH-TIDE TABLE: astronomical height and (where available) surge, at every high
# =====================================================================================
tu = pd.DataFrame([f["properties"] for f in feats])
tu["datetime"]   = pd.to_datetime(tu["datetime"], utc=True)   # all timestamps tz-aware UTC
tu["station_id"] = tu["station_id"].astype(str)               # keep id types consistent

H = tu[tu["tidal_phase"] == "H"].copy()                       # highs only

# astro_ft = the pure astronomical high-tide height, surge stripped out.
#   USGS: observed minus its surge residual.   NOAA: already the astronomical prediction.
H["astro_ft"] = np.where(H.source == "usgs_observed",
                         H.water_level_ft - H.surge_ft,
                         H.water_level_ft)

# surge_ft_h = surge measured AT the high. Known for USGS now; NOAA filled in next cell.
H["surge_ft_h"] = np.where(H.source == "usgs_observed", H.surge_ft, np.nan)

# Restrict to 2025 (by LOCAL calendar date, not UTC) right away, before anything else runs.
# Every QC event is a 2025 date, and everything downstream -- the NOAA surge fetch, HHW,
# percentile ranking -- should only ever see 2025 tide-days. tidal_unified.geojson itself
# spans 2025-2026 (it's built for the separate tidal_analysis.ipynb classification study,
# which wants the extra year); we trim it here rather than touch that shared file.
H["local_date"] = H["datetime"].dt.tz_convert("US/Eastern").dt.date
H = H[pd.to_datetime(H["local_date"]).dt.year == 2025].copy()

H = H[["datetime", "local_date", "station_id", "station_name", "source", "astro_ft", "surge_ft_h"]]
print(H.source.value_counts().to_dict())   # how many 2025 highs per source

{'usgs_observed': 2820, 'noaa_predicted': 1410}


## 3. NOAA surge — fetch observed water level, subtract the prediction

The tide file gives NOAA *predictions* only, so to get surge at the Battery / Kings Point
highs we must fetch the **observed** water level and difference it against the prediction.

Steps in the cell below:
1. Pull 6-minute observed water level from NOAA's API, one month at a time (their API times
   out on long ranges), with automatic retry/backoff on rate-limit / server errors.
2. NOAA returns local wall-clock time (`lst_ldt`, includes daylight saving) and heights above
   the **MLLW** datum. We convert time → UTC and heights → **NAVD88**, using each station's
   own datum offset (`MLLW − NAVD88`, a negative number at NYC gauges). This is the *same*
   conversion `tidal_data_format_conversion.ipynb` applied to the predictions, so observed
   and predicted end up on the same ruler.
3. Cache the result to `noaa_observed_water_levels.csv` so re-runs are instant. Set
   `FORCE_REFRESH = True` at the top to force a fresh pull.

**Heads up:** this cache is ~14 MB (272k 6-minute readings) and is *not* currently gitignored.

In [4]:
# =====================================================================================
# FETCH NOAA OBSERVED WATER LEVELS (cached) -> derive surge at each NOAA high
# =====================================================================================
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

NOAA_API      = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"
NOAA_STATIONS = ["8518750", "8516945"]      # The Battery, Kings Point
YEARS         = [2025]                       # QC events are all 2025 dates; 2026 NOAA highs
                                              # will get NaN surge (harmless, unused downstream)

def _noaa_session():
    """requests session that retries on 429/5xx with exponential backoff (don't hammer the API)."""
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=Retry(
        total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])))
    return s

def fetch_observed(station_id, year, sess):
    """Return [(station_id, local_time_str, observed_ft_MLLW), ...] for one station-year,
    fetched month-by-month to stay under NOAA's response-size limits."""
    recs = []
    for month in range(1, 13):
        b = pd.Timestamp(year, month, 1)
        e = b + pd.offsets.MonthEnd(0)
        r = sess.get(NOAA_API, params={
            "product": "water_level",              # OBSERVED (not "predictions")
            "begin_date": b.strftime("%Y%m%d"),
            "end_date":   e.strftime("%Y%m%d"),
            "datum": "MLLW", "station": station_id,
            "time_zone": "lst_ldt",                # local w/ DST -> we convert to UTC below
            "interval": "6", "units": "english", "format": "json"}, timeout=60)
        r.raise_for_status()
        for row in r.json().get("data", []):
            if row.get("v") not in (None, ""):     # skip gaps (empty value)
                recs.append((station_id, row["t"], float(row["v"])))
    return recs

def mllw_above_navd(station_id):
    """How far MLLW sits above NAVD88 at this station, in feet (negative at NYC gauges).
    Adding this to an MLLW-referenced height converts it to NAVD88 — same as the predictions."""
    r = requests.get(
        f"https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/{station_id}/datums.json",
        params={"units": "english"}, timeout=30)
    r.raise_for_status()
    dd = {d["name"]: float(d["value"]) for d in r.json()["datums"]}
    return dd["MLLW"] - dd["NAVD88"]

# --- Load from cache, or fetch fresh and write the cache ---
if NOAA_CACHE.exists() and not FORCE_REFRESH:
    obs = pd.read_csv(NOAA_CACHE, parse_dates=["datetime"])
    print(f"loaded {len(obs)} cached NOAA observed points")
else:
    sess = _noaa_session()
    recs = [r for sid in NOAA_STATIONS for y in YEARS for r in fetch_observed(sid, y, sess)]
    obs = pd.DataFrame(recs, columns=["station_id", "t", "obs_mllw_ft"])
    # local wall-clock -> US/Eastern (resolve DST) -> UTC. NaT drops the spring-forward gap hour.
    obs["datetime"] = (pd.to_datetime(obs["t"])
                         .dt.tz_localize("US/Eastern", ambiguous="NaT", nonexistent="NaT")
                         .dt.tz_convert("UTC"))
    off = {sid: mllw_above_navd(sid) for sid in NOAA_STATIONS}   # per-station MLLW->NAVD88 offset
    obs["obs_navd_ft"] = obs["obs_mllw_ft"] + obs["station_id"].map(off)
    obs = obs.dropna(subset=["datetime"])[["station_id", "datetime", "obs_navd_ft"]]
    obs.to_csv(NOAA_CACHE, index=False)
    print(f"fetched {len(obs)} NOAA observed points; MLLW->NAVD88 offsets: {off}")

obs["station_id"] = obs["station_id"].astype(str)

loaded 175160 cached NOAA observed points


In [5]:
# --- Attach observed level to each NOAA high, then surge = observed - astronomical -------
# merge_asof matches each predicted-high timestamp to the NEAREST observed 6-min reading
# within 30 minutes (a high may not land exactly on a 6-min mark). Same definition of surge
# as USGS (observed minus astronomical), so NOAA and USGS surge are directly comparable.
noaaH = H[H.source == "noaa_predicted"].sort_values("datetime").copy()
filled = []
for sid, g in noaaH.groupby("station_id"):
    o = obs[obs.station_id == sid].sort_values("datetime")
    m = pd.merge_asof(g.sort_values("datetime"), o[["datetime", "obs_navd_ft"]],
                      on="datetime", direction="nearest", tolerance=pd.Timedelta("30min"))
    m["surge_ft_h"] = m["obs_navd_ft"] - m["astro_ft"]
    filled.append(m.drop(columns=["obs_navd_ft"]))

# Rebuild H with NOAA surge now populated (USGS rows already had it).
H = pd.concat([H[H.source == "usgs_observed"], pd.concat(filled)[H.columns]],
              ignore_index=True)

# Sanity check: how many highs now have a surge value. Some NOAA highs late in 2026 fall
# outside the observed window / have no reading within 30 min -> stay NaN (expected, harmless).
print("surge_ft_h non-null by source:")
print(H.groupby("source")["surge_ft_h"].apply(lambda s: f"{s.notna().sum()}/{len(s)}"))

surge_ft_h non-null by source:
source
noaa_predicted    1410/1410
usgs_observed     2820/2820
Name: surge_ft_h, dtype: str


## 4. Reduce to the day's **highest high water (HHW)** and rank it within the year

There are ~2 highs per day, and with no event time we can't know which one (if either)
caused the flood — so we take the **higher** of the day's two highs (the flood-relevant
worst case) and the surge measured at that same high.

Then, to answer "was this an unusually high tide day?", we convert each day's HHW into a
**percentile within that station's own full year** of daily HHW values. Percentile is
*datum-invariant* (a constant offset doesn't change ranking), so it's a clean, comparable
measure of "how high was this day relative to a typical day here":
- percentile ≈ 1.0 → one of the highest tides of the year (spring / king tide)
- percentile ≈ 0.5 → a completely ordinary day
- a *random* day has an expected percentile of 0.5 — this is the null we test against later.

In [6]:
# =====================================================================================
# DAILY HIGHEST HIGH WATER (HHW) + within-year percentile rank
# =====================================================================================
# For each (station, day) keep the row with the MAX astronomical height = the higher high.
# idxmax gives that row's index; we carry along the surge measured at that same high.
idx = H.groupby(["station_id", "local_date"])["astro_ft"].idxmax()
hhw = H.loc[idx, ["station_id", "station_name", "local_date",
                  "astro_ft", "surge_ft_h"]].copy()

# Rank within each station's own year. pct=True -> percentile in [0, 1]. NaNs are skipped.
hhw["astro_pctile"] = hhw.groupby("station_id")["astro_ft"].rank(pct=True)     # tide magnitude
hhw["surge_pctile"] = hhw.groupby("station_id")["surge_ft_h"].rank(pct=True)   # surge magnitude

print(f"{len(hhw)} station-days of HHW built")
# Per-station spread of astronomical highs and surge, for a gut check on the numbers.
hhw.groupby("station_name")[["astro_ft", "surge_ft_h"]].describe().round(2).T

2190 station-days of HHW built


station_name      EAST ROCKAWAY INLET AT ATLANTIC BEACH NY  \
astro_ft   count                                    365.00   
           mean                                       2.46   
           std                                        0.58   
           min                                        0.83   
           25%                                        2.06   
           50%                                        2.43   
           75%                                        2.90   
           max                                        3.67   
surge_ft_h count                                    365.00   
           mean                                      -0.02   
           std                                        0.49   
           min                                       -2.23   
           25%                                       -0.23   
           50%                                       -0.04   
           75%                                        0.22   
           max                                        2.31   

station_name      GREAT KILLS HARBOR AT GREAT KILLS NY  \
astro_ft   count                                365.00   
           mean                                   2.88   
           std                                    0.65   
           min                                    1.30   
           25%                                    2.43   
           50%                                    2.81   
           75%                                    3.37   
           max                                    4.19   
surge_ft_h count                                365.00   
           mean                                  -0.02   
           std                                    0.55   
           min                                   -2.46   
           25%                                   -0.25   
           50%                                   -0.04   
           75%                                    0.27   
           max                                    2.29   

station_name      JAMAICA BAY AT INWOOD NY  Kings Point  \
astro_ft   count                    365.00       365.00   
           mean                       3.08         3.62   
           std                        0.67         0.66   
           min                        1.50         2.10   
           25%                        2.62         3.17   
           50%                        2.99         3.54   
           75%                        3.58         4.01   
           max                        4.44         5.40   
surge_ft_h count                    365.00       365.00   
           mean                      -0.03         0.20   
           std                        0.48         0.71   
           min                       -2.21        -3.15   
           25%                       -0.26        -0.06   
           50%                       -0.03         0.26   
           75%                        0.21         0.60   
           max                        2.01         2.42   

station_name      ROCKAWAY INLET NEAR FLOYD BENNETT FIELD NY  The Battery  
astro_ft   count                                      365.00       365.00  
           mean                                         3.02         2.24  
           std                                          0.64         0.54  
           min                                          1.25         0.83  
           25%                                          2.57         1.88  
           50%                                          2.95         2.21  
           75%                                          3.50         2.59  
           max                                          4.32         3.57  
surge_ft_h count                                      365.00       365.00  
           mean                                        -0.02         0.38  
           std                                          0.50         0.60  
           min                            

## 5. Classify every QC event

Join each event to its station-day HHW row and bucket it:

| bucket | meaning |
|---|---|
| **tidal (high astronomical)** | HHW in the top 20% of the year (`astro_pctile ≥ 0.80`) — a king/spring-tide day |
| **surge** | ≥ 1 ft of positive surge at the high — weather pushed water up |
| **compound (tide + surge)** | both of the above |
| **neither** | ordinary tide, no notable surge (and, if also no rain, genuinely unexplained) |

A flood on a NOAA station-day missing its surge value is judged on tide alone (surge treated
as "not elevated"); this only ever makes the tidal/surge buckets *conservative*.

In [7]:
# =====================================================================================
# BUCKET EACH EVENT: tidal / surge / compound / neither
# =====================================================================================
# Left-join events to their (station, date) HHW row. how="left" keeps every event even if a
# tide-day is somehow missing (should be 0 — checked in the print).
m = ev.merge(hhw[["station_id", "local_date", "astro_ft", "surge_ft_h",
                  "astro_pctile", "surge_pctile"]],
             left_on=["station_id", "date"], right_on=["station_id", "local_date"],
             how="left")

def classify(r):
    if pd.isna(r.astro_ft):
        return "no tide data"                                   # event day had no tide record
    tide  = r.astro_pctile >= ASTRO_PCTILE_THRESHOLD            # unusually high astronomical tide?
    surge = pd.notna(r.surge_ft_h) and r.surge_ft_h >= SURGE_FT_THRESHOLD   # notable surge?
    if tide and surge: return "compound (tide+surge)"
    if tide:           return "tidal (high astronomical)"
    if surge:          return "surge"
    return "neither"

m["tide_class"] = m.apply(classify, axis=1)
print(f"{m.astro_ft.isna().sum()} events with no matching tide-day (expect 0)")
m["tide_class"].value_counts()

0 events with no matching tide-day (expect 0)


tide_class
neither                      168
tidal (high astronomical)     42
surge                         15
compound (tide+surge)          1
Name: count, dtype: int64

## 6. Results — cross-tab against rain, and the enrichment test

Two views:
1. **Cross-tab** of tide-class × rain-flag over all 227 events — the whole picture at a glance.
2. **Enrichment test** on the no-rain events — the honest statistical check. If these no-rain
   floods were tidally driven, their HHW percentiles should skew **high**: mean well above the
   0.50 random-day null, and far more than 20% of them above the 0.80 threshold. If instead the
   mean sits near or below 0.50, the events are *not* preferentially on high-tide days.

Remember the counts here are **events**, not sensors.

In [8]:
# =====================================================================================
# CROSS-TAB + ENRICHMENT TEST
# =====================================================================================
print(f"=== All {len(m)} events: tide class x rain flag (counts are EVENTS) ===")
print(pd.crosstab(m.tide_class, m.rain_flag.map({True: "rain", False: "no-rain"}),
                  margins=True))

# Focus on events with NO rain driver — the tidal candidates.
no_rain = m[~m.rain_flag]
print(f"\n=== No-rain events (n={len(no_rain)}) by tide class ===")
print(no_rain.tide_class.value_counts())

# Enrichment test vs the uniform null (random day => mean percentile 0.50, 20% above 0.80).
print("\n=== Enrichment test on no-rain events ===")
ap = no_rain.astro_pctile.dropna()
print(f"mean astronomical HHW percentile : {ap.mean():.3f}   (null = 0.500)")
print(f"share above {ASTRO_PCTILE_THRESHOLD:.2f} percentile     : "
      f"{(ap >= ASTRO_PCTILE_THRESHOLD).mean():.1%}   (null = 20.0%)")
sp = no_rain.surge_ft_h.dropna()
print(f"median surge at the high          : {sp.median():+.2f} ft")
print(f"share with surge >= {SURGE_FT_THRESHOLD:.1f} ft         : "
      f"{(sp >= SURGE_FT_THRESHOLD).mean():.1%}")

=== All 226 events: tide class x rain flag (counts are EVENTS) ===
rain_flag                  no-rain  rain  All
tide_class                                   
compound (tide+surge)            1     0    1
neither                        129    39  168
surge                            8     7   15
tidal (high astronomical)       26    16   42
All                            164    62  226

=== No-rain events (n=164) by tide class ===
tide_class
neither                      129
tidal (high astronomical)     26
surge                          8
compound (tide+surge)          1
Name: count, dtype: int64

=== Enrichment test on no-rain events ===
mean astronomical HHW percentile : 0.380   (null = 0.500)
share above 0.80 percentile     : 16.5%   (null = 20.0%)
median surge at the high          : +0.12 ft
share with surge >= 1.0 ft         : 5.5%


In [9]:
# =====================================================================================
# PER-SENSOR ROLLUP (no-rain events only) — is any single sensor tidal even if the whole set isn't?
# =====================================================================================
roll = (no_rain.groupby(["sensor_name", "station_name"])
        .agg(n_events=("tide_class", "size"),
             mean_astro_pctile=("astro_pctile", "mean"),   # >0.5 would hint tidal, <0.5 anti-tidal
             pct_tidal=("tide_class",
                        lambda s: s.str.startswith(("tidal", "compound")).mean()),
             median_surge_ft=("surge_ft_h", "median"))
        .round(3).sort_values("mean_astro_pctile", ascending=False))
roll

,,n_events,mean_astro_pctile,pct_tidal,median_surge_ft
sensor_name,station_name,,,,
BX - Tier St/William Ave,Kings Point,8,0.557,0.375,0.074
SI - Hylan Blvd/ Jefferson Blvd,GREAT KILLS HARBOR AT GREAT KILLS NY,8,0.516,0.250,-0.003
SI - Minthorne St/ Victory Blvd,The Battery,29,0.486,0.276,0.369
SI - Baden Pl/ Mapleton Ave,GREAT KILLS HARBOR AT GREAT KILLS NY,14,0.388,0.214,0.251
SI - Grimsby St/ Mapleton Ave,GREAT KILLS HARBOR AT GREAT KILLS NY,34,0.370,0.088,0.030
SI - McLaughlin St/Agnes Pl,GREAT KILLS HARBOR AT GREAT KILLS NY,24,0.357,0.167,0.004
Q - Beach 49th St/Rockaway Beach Blvd,JAMAICA BAY AT INWOOD NY,6,0.338,0.167,-0.162
SI - Bedford Ave/Kiswick St,GREAT KILLS HARBOR AT GREAT KILLS NY,6,0.318,0.167,-0.015
SI - Snug Harbor Rd / Kissel Ave,The Battery,12,0.297,0.167,0.400


**How to read this rollup.** One row per sensor, over its no-rain events only.
`mean_astro_pctile` is the key column: **above 0.5** means that sensor's missing floods tend
to fall on higher-than-average tide days (tidal-leaning); **below 0.5** means they fall on
lower-than-average tide days (anti-tidal). `pct_tidal` is the fraction of its events that
cleared the king-tide/compound bucket. Sorted highest-first, so if any sensor were tidal it
would be at the top — watch whether even the top sensor clears 0.5.

In [10]:
# =====================================================================================
# EXPORT the full per-event table for manual inspection / editing next session
# =====================================================================================
cols = ["date", "sensor_name", "station_name", "dist_km", "rain_flag",
        "astro_ft", "astro_pctile", "surge_ft_h", "tide_class"]
m_out = m[cols].sort_values(["sensor_name", "date"])
m_out.to_csv(BASE / "tidal_corroboration_events.csv", index=False)
print(f"wrote tidal_corroboration_events.csv ({len(m_out)} rows)")
m_out.head(20)

wrote tidal_corroboration_events.csv (226 rows)


,date,sensor_name,station_name,dist_km,rain_flag,astro_ft,astro_pctile,surge_ft_h,tide_class
40,2025-03-02,BX - Tier St/William Ave,Kings Point,4.68,False,4.387,0.873973,-0.582,tidal (high astronomical)
41,2025-03-05,BX - Tier St/William Ave,Kings Point,4.68,False,3.836,0.660274,0.045,neither
50,2025-03-09,BX - Tier St/William Ave,Kings Point,4.68,False,2.838,0.093151,-0.880,neither
58,2025-03-16,BX - Tier St/William Ave,Kings Point,4.68,True,3.427,0.417808,0.417,neither
63,2025-03-21,BX - Tier St/William Ave,Kings Point,4.68,True,2.656,0.049315,1.297,surge
84,2025-04-05,BX - Tier St/William Ave,Kings Point,4.68,True,2.848,0.095890,0.540,neither
111,2025-04-25,BX - Tier St/William Ave,Kings Point,4.68,False,4.540,0.895890,-0.200,tidal (high astronomical)
113,2025-04-26,BX - Tier St/William Ave,Kings Point,4.68,True,4.989,0.961644,-0.127,tidal (high astronomical)
126,2025-05-04,BX - Tier St/William Ave,Kings Point,4.68,False,2.937,0.130137,0.366,neither
131,2025-05-08,BX - Tier St/William Ave,Kings Point,4.68,True,3.671,0.564384,0.190,neither


## What this shows

The day-level tide/surge test comes back **negative for the extreme-tide/surge hypothesis**:
the 164 no-rain missing events are not disproportionately king-tide or storm-surge days.

- Mean astronomical HHW percentile for no-rain events: **0.380** (null = 0.500) — slightly
  *below* an ordinary day, not above. Only **16.5%** land in the top-20% "king tide" bucket
  (null = 20%).
- Surge is negligible: median **+0.12 ft**, only **5.5%** at/above the 1 ft surge threshold.
- Per-sensor rollup: **9 of the 11 sensors** sit clearly below the 0.50 null (0.221–0.486).
  Two — `BX - Tier St/William Ave` (0.557) and `SI - Hylan Blvd/Jefferson Blvd` (0.516) — sit
  just above it, but each on only **8** no-rain events, small enough that this is plausibly
  noise rather than a real per-sensor tidal lean, not something to build a claim on.
- So **129 of 164** no-rain events land in "neither": ordinary tide, no surge.

**What this rules out, and what it doesn't.** This result rules out *extreme* (king) tides
and storm surge as the driver of these no-rain events. It does **not** rule out *ordinary*,
everyday tidal flooding: a sensor that floods at every routine high tide (because it simply
sits at low elevation) would score right around the 0.50 null — exactly where most of these
events land. So this test is actually *consistent with* routine tidal flooding, not evidence
against it. Telling "ordinary tidal" apart from "not tidal at all" needs the **phase test**
(lag to the high-tide hour), which needs sub-daily timestamps this day-level QC log doesn't
have.

**More importantly — this test says nothing about whether the graph bump was a real flood or
sensor noise.** It only ever works from the QC log's *dates*; it never looks at the shape of
the depth trace itself. A negative tide/surge/rain result narrows down *which physical driver*
would explain a genuinely-missing event — it does not confirm the event happened at all.
Settling "was this bump real or an artifact" is a different kind of check, most directly by
looking at the raw depth trace's shape (rise → peak → recede, vs. an isolated blip) — that has
to happen before concluding an event should have been in the public dataset.

**Bottom line:** these missing events are not extreme-tide or surge events. Whether they are
ordinary tidal floods, rain the automated check missed (hyperlocal rain is patchy and the
nearest station can be 3–10 km away), or a graph bump that isn't really a flood at all, is
still open — and answering it needs the raw depth graphs, not more day-level corroboration.

## Caveats (method limitations to keep in mind while editing)

- **Day-level, not phase-level.** This tests whether a flood fell on a high-tide-*magnitude*
  day, not whether it occurred at the *hour* of high tide. A sensor that floods at every
  ordinary high tide scores near the 0.50 null here — indistinguishable from "not tidal" by
  this test alone. See the "What this shows" cell above. Recovering sub-daily timestamps
  from the raw depth graphs would enable the stronger lag-to-high-tide (Rayleigh) test used
  in `tidal_analysis.ipynb`.
- **This test cannot confirm an event happened at all.** It only ever works from the QC
  log's dates, never the shape of the depth trace. A negative result narrows down *which
  physical driver* would explain a genuinely-missing event; whether the graph bump was a
  real flood or sensor noise is a separate question, answered by looking at the raw trace.
- **Thresholds are choices.** `ASTRO_PCTILE_THRESHOLD` (0.80) and `SURGE_FT_THRESHOLD` (1.0 ft)
  set the buckets; the enrichment test is threshold-free and is the more robust reading.
- **Rain flag is the automated hyperlocal check** (rain at the nearest station, event day or
  the day before), built in `Weather_Data_Scrape.ipynb` — not the manual QC `Weather` column.
  It's still an imperfect instrument: NYC rain is patchy and the nearest station can be
  3–10 km away, so a real hyperlocal downpour can be missed.
- **NOAA vs USGS surge definition.** USGS surge is observed − UTide reconstruction; NOAA surge is
  observed − NOAA harmonic prediction. Both are astronomical residuals but from different harmonic
  fits — comparable in spirit, not identical in construction.